In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Supplementary Domain-Probing Analysis (FIXED)

**Question.** How strongly is corpus identity linearly decodable from the utterance-level handcrafted representation versus the utterance-level emotion2vec representation?

This notebook is a supplementary diagnostic analysis. It does **not** establish that corpus-specific information causes the fusion-performance differences, and it does **not** causally isolate language, recording condition, speaking style, or any other corpus factor.

Key safeguards in this fixed version:

1. Handcrafted features are converted back to their **pre-scaling/raw feature space** using each corpus's saved `scaler_hc.pkl`. This avoids comparing per-corpus-standardized handcrafted features against unscaled emotion2vec embeddings.
2. Handcrafted and emotion2vec samples are aligned explicitly by `uid`.
3. Corpus probing uses **speaker-disjoint 5-fold StratifiedGroupKFold**.
4. Scaling for the probe is fitted **only on the outer training fold** and then applied to the held-out speakers.
5. Logistic regression is deterministic here; results are therefore summarized across the **five held-out speaker folds**, not across artificial random-seed repetitions.
6. An optional **emotion-conditioned probe** repeats corpus prediction separately within each emotion class to reduce confounding from different emotion-class distributions across corpora.

Interpretation should be phrased as **linear decodability of corpus identity**, not proof of causal "corpus-specific bias."


In [2]:
import warnings
warnings.filterwarnings("ignore")

import pickle
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

N_SPLITS = 5
CV_RANDOM_STATE = 42

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

HC_DIR = BASE_PROJECT / "processed_intra_features_hc_noaug"
E2V_DIR = BASE_PROJECT / "processed_intra_features_e2v_plus_base"

OUT_DIR = BASE_PROJECT / "results_domain_probing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]
CORPUS_TO_ID = {"emodb": 0, "ravdess": 1, "resd": 2}
ID_TO_CORPUS = {v: k for k, v in CORPUS_TO_ID.items()}

EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]


In [3]:
def load_aligned_dataset(dataset_name):
    """
    Load handcrafted and emotion2vec utterance-level features and align them by UID.

    IMPORTANT:
    X_hc_* files in processed_intra_features_hc_noaug were already transformed
    by a corpus-specific scaler in the original intra-corpus pipeline.
    We therefore recover the pre-scaling handcrafted values using scaler_hc.pkl
    before performing the domain probe.

    The domain-probe scaler is later fitted only on each outer training fold.
    """
    hc_ds = HC_DIR / dataset_name
    e2v_ds = E2V_DIR / dataset_name

    with open(hc_ds / "scaler_hc.pkl", "rb") as f:
        original_hc_scaler = pickle.load(f)

    rows = []

    for split in ["train", "val", "test"]:
        X_hc_scaled = np.load(hc_ds / f"X_hc_{split}.npy").astype(np.float32)
        X_e2v = np.load(e2v_ds / f"X_e2v_{split}.npy").astype(np.float32)

        meta_hc = pd.read_csv(hc_ds / f"meta_{split}.csv").reset_index(drop=True)
        meta_e2v = pd.read_csv(e2v_ds / f"meta_{split}.csv").reset_index(drop=True)

        assert len(X_hc_scaled) == len(meta_hc), f"HC length mismatch: {dataset_name} {split}"
        assert len(X_e2v) == len(meta_e2v), f"e2v length mismatch: {dataset_name} {split}"
        assert "uid" in meta_hc.columns and "uid" in meta_e2v.columns
        assert "speaker" in meta_hc.columns and "speaker" in meta_e2v.columns
        assert "emotion" in meta_hc.columns and "emotion" in meta_e2v.columns

        # The no-augmentation directory should contain original utterances only.
        if "is_augmented" in meta_hc.columns:
            assert not meta_hc["is_augmented"].fillna(False).astype(bool).any(), (
                f"Augmented samples found in {dataset_name} {split}"
            )

        # Recover raw/pre-scaling handcrafted features.
        X_hc_raw = original_hc_scaler.inverse_transform(X_hc_scaled).astype(np.float32)

        hc_uid_to_idx = {str(uid): i for i, uid in enumerate(meta_hc["uid"])}
        e2v_uid_to_idx = {str(uid): i for i, uid in enumerate(meta_e2v["uid"])}

        common_uids = [str(uid) for uid in meta_hc["uid"] if str(uid) in e2v_uid_to_idx]
        dropped_hc = len(meta_hc) - len(common_uids)
        dropped_e2v = len(meta_e2v) - len(common_uids)

        if dropped_hc or dropped_e2v:
            print(
                f"WARNING {dataset_name} {split}: using UID intersection; "
                f"dropped HC={dropped_hc}, e2v={dropped_e2v}"
            )

        for uid in common_uids:
            ih = hc_uid_to_idx[uid]
            ie = e2v_uid_to_idx[uid]

            # Metadata agreement checks.
            spk_h = str(meta_hc.loc[ih, "speaker"])
            spk_e = str(meta_e2v.loc[ie, "speaker"])
            emo_h = str(meta_hc.loc[ih, "emotion"])
            emo_e = str(meta_e2v.loc[ie, "emotion"])

            assert spk_h == spk_e, f"Speaker mismatch for UID={uid}"
            assert emo_h == emo_e, f"Emotion mismatch for UID={uid}"

            rows.append({
                "dataset": dataset_name,
                "corpus_id": CORPUS_TO_ID[dataset_name],
                "uid": uid,
                "speaker": f"{dataset_name}_{spk_h}",
                "emotion": emo_h,
                "split_original": split,
                "X_hc": X_hc_raw[ih],
                "X_e2v": X_e2v[ie],
            })

    return rows


all_rows = []
for ds in DATASETS:
    ds_rows = load_aligned_dataset(ds)
    all_rows.extend(ds_rows)
    print(
        f"{ds}: N={len(ds_rows)}, "
        f"speakers={len(set(r['speaker'] for r in ds_rows))}"
    )

meta = pd.DataFrame(
    [{k: v for k, v in r.items() if k not in ["X_hc", "X_e2v"]} for r in all_rows]
)
X_hc_all = np.stack([r["X_hc"] for r in all_rows]).astype(np.float32)
X_e2v_all = np.stack([r["X_e2v"] for r in all_rows]).astype(np.float32)

speaker_all = meta["speaker"].to_numpy()
corpus_all = meta["corpus_id"].to_numpy(dtype=np.int64)

print("\nAligned pooled dataset")
print("N:", len(meta))
print("HC shape:", X_hc_all.shape)
print("e2v shape:", X_e2v_all.shape)
print("Corpus counts:", meta["dataset"].value_counts().to_dict())
print("Emotion counts:", meta["emotion"].value_counts().to_dict())

assert len(meta) == len(X_hc_all) == len(X_e2v_all)
assert meta["uid"].notna().all()


emodb: N=718, speakers=10
ravdess: N=1056, speakers=24
resd: N=1198, speakers=50

Aligned pooled dataset
N: 2972
HC shape: (2972, 548)
e2v shape: (2972, 768)
Corpus counts: {'resd': 1198, 'ravdess': 1056, 'emodb': 718}
Emotion counts: {'angry': 551, 'fear': 538, 'happy': 528, 'disgust': 483, 'sad': 479, 'neutral': 393}


In [4]:
def make_outer_splits(y, groups, n_splits=N_SPLITS, random_state=CV_RANDOM_STATE):
    cv = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )
    splits = list(cv.split(np.zeros(len(y)), y, groups=groups))

    # Safety checks: speaker-disjoint and every corpus represented in train/test.
    all_classes = set(np.unique(y))
    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        train_groups = set(groups[train_idx])
        test_groups = set(groups[test_idx])

        assert train_groups.isdisjoint(test_groups), f"Speaker leakage in fold {fold}"
        assert set(np.unique(y[train_idx])) == all_classes, f"Missing train corpus in fold {fold}"
        assert set(np.unique(y[test_idx])) == all_classes, f"Missing test corpus in fold {fold}"

    return splits


outer_splits = make_outer_splits(corpus_all, speaker_all)

for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    print(
        f"Fold {fold}: train={len(train_idx)}, test={len(test_idx)}, "
        f"test corpus counts={pd.Series(corpus_all[test_idx]).map(ID_TO_CORPUS).value_counts().to_dict()}"
    )


Fold 1: train=2338, test=634, test corpus counts={'emodb': 280, 'resd': 222, 'ravdess': 132}
Fold 2: train=2395, test=577, test corpus counts={'resd': 255, 'ravdess': 176, 'emodb': 146}
Fold 3: train=2403, test=569, test corpus counts={'ravdess': 264, 'resd': 233, 'emodb': 72}
Fold 4: train=2377, test=595, test corpus counts={'ravdess': 308, 'resd': 214, 'emodb': 73}
Fold 5: train=2375, test=597, test corpus counts={'resd': 274, 'ravdess': 176, 'emodb': 147}


In [5]:
def run_linear_domain_probe(X, y, groups, feature_name, splits):
    rows = []
    oof_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits, start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        clf = LogisticRegression(
            C=1.0,
            max_iter=3000,
            solver="lbfgs",
            class_weight="balanced",
        )
        clf.fit(X_train_s, y_train)
        pred = clf.predict(X_test_s)

        rows.append({
            "feature": feature_name,
            "fold": fold_idx,
            "accuracy": accuracy_score(y_test, pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro"),
            "n_test": len(test_idx),
        })

        for idx, yt, yp in zip(test_idx, y_test, pred):
            oof_rows.append({
                "feature": feature_name,
                "fold": fold_idx,
                "row_index": int(idx),
                "true_corpus_id": int(yt),
                "pred_corpus_id": int(yp),
                "true_corpus": ID_TO_CORPUS[int(yt)],
                "pred_corpus": ID_TO_CORPUS[int(yp)],
            })

        print(
            f"[{feature_name}] fold {fold_idx}: "
            f"acc={rows[-1]['accuracy']:.4f}, "
            f"bal_acc={rows[-1]['balanced_accuracy']:.4f}, "
            f"macro_f1={rows[-1]['macro_f1']:.4f}"
        )

    return pd.DataFrame(rows), pd.DataFrame(oof_rows)


hc_results, hc_oof = run_linear_domain_probe(
    X_hc_all, corpus_all, speaker_all, "Handcrafted", outer_splits
)
e2v_results, e2v_oof = run_linear_domain_probe(
    X_e2v_all, corpus_all, speaker_all, "emotion2vec", outer_splits
)

probe_results = pd.concat([hc_results, e2v_results], ignore_index=True)
probe_oof = pd.concat([hc_oof, e2v_oof], ignore_index=True)

probe_results.to_csv(OUT_DIR / "domain_probe_all_results_FIXED.csv", index=False)
probe_oof.to_csv(OUT_DIR / "domain_probe_oof_predictions_FIXED.csv", index=False)

display(probe_results)


[Handcrafted] fold 1: acc=0.9890, bal_acc=0.9901, macro_f1=0.9892
[Handcrafted] fold 2: acc=0.9931, bal_acc=0.9928, macro_f1=0.9928
[Handcrafted] fold 3: acc=0.9930, bal_acc=0.9911, macro_f1=0.9912
[Handcrafted] fold 4: acc=0.9983, bal_acc=0.9984, macro_f1=0.9970
[Handcrafted] fold 5: acc=0.9916, bal_acc=0.9918, macro_f1=0.9910
[emotion2vec] fold 1: acc=0.7965, bal_acc=0.8190, macro_f1=0.8036
[emotion2vec] fold 2: acc=0.7400, bal_acc=0.7483, macro_f1=0.7359
[emotion2vec] fold 3: acc=0.8225, bal_acc=0.7954, macro_f1=0.7677
[emotion2vec] fold 4: acc=0.7765, bal_acc=0.7219, macro_f1=0.7062
[emotion2vec] fold 5: acc=0.7822, bal_acc=0.7974, macro_f1=0.7781


,feature,fold,accuracy,balanced_accuracy,macro_f1,n_test
0,Handcrafted,1,0.988959,0.990112,0.989232,634
1,Handcrafted,2,0.993068,0.992819,0.992819,577
2,Handcrafted,3,0.992970,0.991079,0.991239,569
3,Handcrafted,4,0.998319,0.998442,0.996952,595
4,Handcrafted,5,0.991625,0.991815,0.990964,597
5,emotion2vec,1,0.796530,0.818994,0.803586,634
6,emotion2vec,2,0.740035,0.748270,0.735908,577
7,emotion2vec,3,0.822496,0.795391,0.767722,569
8,emotion2vec,4,0.776471,0.721912,0.706175,595
9,emotion2vec,5,0.782245,0.797448,0.778108,597


In [6]:
# Summary across the five held-out speaker folds.
summary = (
    probe_results
    .groupby("feature")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_sd_across_folds=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_sd_across_folds=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_sd_across_folds=("macro_f1", "std"),
        n_outer_folds=("fold", "nunique"),
    )
    .reset_index()
)

summary.to_csv(OUT_DIR / "domain_probe_summary_FIXED.csv", index=False)
display(summary)

# Paired fold-wise difference: Handcrafted minus emotion2vec.
paired = (
    hc_results[["fold", "accuracy", "balanced_accuracy", "macro_f1"]]
    .merge(
        e2v_results[["fold", "accuracy", "balanced_accuracy", "macro_f1"]],
        on="fold",
        suffixes=("_hc", "_e2v"),
    )
)

for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
    paired[f"{metric}_difference_hc_minus_e2v"] = (
        paired[f"{metric}_hc"] - paired[f"{metric}_e2v"]
    )

paired.to_csv(
    OUT_DIR / "domain_probe_paired_fold_differences_FIXED.csv",
    index=False,
)
display(paired)

print("\nInterpretation guardrail:")
print(
    "A higher Handcrafted score means corpus identity is more linearly decodable "
    "from the handcrafted representation under this matched speaker-disjoint probe. "
    "It does NOT by itself prove that corpus information caused the SER fusion result."
)


,feature,accuracy_mean,accuracy_sd_across_folds,balanced_accuracy_mean,balanced_accuracy_sd_across_folds,macro_f1_mean,macro_f1_sd_across_folds,n_outer_folds
0,Handcrafted,0.992988,0.003410,0.992853,0.003278,0.992241,0.002925,5
1,emotion2vec,0.783555,0.030127,0.776403,0.039932,0.758300,0.037916,5


,fold,accuracy_hc,balanced_accuracy_hc,macro_f1_hc,accuracy_e2v,balanced_accuracy_e2v,macro_f1_e2v,accuracy_difference_hc_minus_e2v,balanced_accuracy_difference_hc_minus_e2v,macro_f1_difference_hc_minus_e2v
0,1,0.988959,0.990112,0.989232,0.796530,0.818994,0.803586,0.192429,0.171118,0.185646
1,2,0.993068,0.992819,0.992819,0.740035,0.748270,0.735908,0.253033,0.244549,0.256912
2,3,0.992970,0.991079,0.991239,0.822496,0.795391,0.767722,0.170475,0.195687,0.223517
3,4,0.998319,0.998442,0.996952,0.776471,0.721912,0.706175,0.221849,0.276530,0.290777
4,5,0.991625,0.991815,0.990964,0.782245,0.797448,0.778108,0.209380,0.194367,0.212856



Interpretation guardrail:
A higher Handcrafted score means corpus identity is more linearly decodable from the handcrafted representation under this matched speaker-disjoint probe. It does NOT by itself prove that corpus information caused the SER fusion result.


In [7]:
# Aggregate out-of-fold confusion matrices across all five folds.
confusion_rows = []

for feature_name, df_oof in probe_oof.groupby("feature"):
    cm = confusion_matrix(
        df_oof["true_corpus_id"],
        df_oof["pred_corpus_id"],
        labels=[0, 1, 2],
    )

    print(f"\n{feature_name} OOF confusion matrix:")
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{ID_TO_CORPUS[i]}" for i in range(3)],
        columns=[f"pred_{ID_TO_CORPUS[i]}" for i in range(3)],
    )
    display(cm_df)

    for true_id in range(3):
        for pred_id in range(3):
            confusion_rows.append({
                "feature": feature_name,
                "true_corpus": ID_TO_CORPUS[true_id],
                "pred_corpus": ID_TO_CORPUS[pred_id],
                "count": int(cm[true_id, pred_id]),
            })

pd.DataFrame(confusion_rows).to_csv(
    OUT_DIR / "domain_probe_oof_confusion_FIXED.csv",
    index=False,
)



Handcrafted OOF confusion matrix:


,pred_emodb,pred_ravdess,pred_resd
true_emodb,711,1,6
true_ravdess,0,1056,0
true_resd,10,4,1184



emotion2vec OOF confusion matrix:


,pred_emodb,pred_ravdess,pred_resd
true_emodb,515,69,134
true_ravdess,75,936,45
true_resd,224,96,878


## Optional emotion-conditioned sensitivity analysis

The main probe above uses the full mapped six-class corpus distribution. Because emotion-class frequencies differ across corpora, part of corpus predictability could in principle be associated with emotion-distribution differences.

The following sensitivity analysis predicts corpus identity **separately within each emotion class**. This removes emotion class itself as a between-sample discriminator, although other corpus characteristics (language, recording conditions, lexical content, speaking style, speaker population, etc.) remain jointly confounded.

This is still a diagnostic probe, not a causal decomposition.


In [8]:
def run_emotion_conditioned_probes():
    rows = []

    for emotion in EMOTIONS:
        mask = (meta["emotion"].to_numpy() == emotion)
        idx = np.where(mask)[0]

        y = corpus_all[idx]
        groups = speaker_all[idx]
        X_hc = X_hc_all[idx]
        X_e2v = X_e2v_all[idx]

        # Need enough speaker groups from each corpus for five group-stratified folds.
        group_frame = pd.DataFrame({
            "group": groups,
            "corpus": y,
        }).drop_duplicates()
        group_counts = group_frame.groupby("corpus")["group"].nunique()

        if len(group_counts) < 3 or group_counts.min() < N_SPLITS:
            print(
                f"Skipping {emotion}: insufficient speaker groups per corpus "
                f"for {N_SPLITS}-fold CV: {group_counts.to_dict()}"
            )
            continue

        splits = make_outer_splits(
            y,
            groups,
            n_splits=N_SPLITS,
            random_state=CV_RANDOM_STATE,
        )

        for feature_name, X in [
            ("Handcrafted", X_hc),
            ("emotion2vec", X_e2v),
        ]:
            result_df, _ = run_linear_domain_probe(
                X, y, groups, feature_name, splits
            )
            result_df["emotion"] = emotion
            rows.append(result_df)

    if not rows:
        return pd.DataFrame(), pd.DataFrame()

    all_results = pd.concat(rows, ignore_index=True)

    summary_by_emotion = (
        all_results
        .groupby(["emotion", "feature"])
        .agg(
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_sd_across_folds=("macro_f1", "std"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_sd_across_folds=("balanced_accuracy", "std"),
            n_outer_folds=("fold", "nunique"),
        )
        .reset_index()
    )

    return all_results, summary_by_emotion


emotion_probe_results, emotion_probe_summary = run_emotion_conditioned_probes()

if len(emotion_probe_results):
    emotion_probe_results.to_csv(
        OUT_DIR / "domain_probe_by_emotion_all_results_FIXED.csv",
        index=False,
    )
    emotion_probe_summary.to_csv(
        OUT_DIR / "domain_probe_by_emotion_summary_FIXED.csv",
        index=False,
    )
    display(emotion_probe_summary)

print("\nSaved fixed domain-probing outputs to:", OUT_DIR)


[Handcrafted] fold 1: acc=0.9281, bal_acc=0.9428, macro_f1=0.9254
[Handcrafted] fold 2: acc=0.9519, bal_acc=0.9573, macro_f1=0.9469
[Handcrafted] fold 3: acc=1.0000, bal_acc=1.0000, macro_f1=1.0000
[Handcrafted] fold 4: acc=0.9820, bal_acc=0.9804, macro_f1=0.9788
[Handcrafted] fold 5: acc=1.0000, bal_acc=1.0000, macro_f1=1.0000
[emotion2vec] fold 1: acc=0.6763, bal_acc=0.7141, macro_f1=0.6691
[emotion2vec] fold 2: acc=0.7981, bal_acc=0.8083, macro_f1=0.7948
[emotion2vec] fold 3: acc=0.9341, bal_acc=0.9444, macro_f1=0.9180
[emotion2vec] fold 4: acc=0.9189, bal_acc=0.9112, macro_f1=0.9084
[emotion2vec] fold 5: acc=0.8962, bal_acc=0.8907, macro_f1=0.8892
[Handcrafted] fold 1: acc=0.9677, bal_acc=0.9750, macro_f1=0.9692
[Handcrafted] fold 2: acc=0.9459, bal_acc=0.9667, macro_f1=0.9304
[Handcrafted] fold 3: acc=0.9789, bal_acc=0.9785, macro_f1=0.9790
[Handcrafted] fold 4: acc=1.0000, bal_acc=1.0000, macro_f1=1.0000
[Handcrafted] fold 5: acc=0.9677, bal_acc=0.9677, macro_f1=0.9672
[emotion2v

,emotion,feature,macro_f1_mean,macro_f1_sd_across_folds,balanced_accuracy_mean,balanced_accuracy_sd_across_folds,n_outer_folds
0,angry,Handcrafted,0.970206,0.033180,0.976096,0.025607,5
1,angry,emotion2vec,0.835869,0.105301,0.853756,0.092803,5
2,disgust,Handcrafted,0.969141,0.025262,0.977576,0.013472,5
3,disgust,emotion2vec,0.812306,0.026382,0.836118,0.035286,5
4,fear,Handcrafted,0.953582,0.049429,0.971955,0.023120,5
5,fear,emotion2vec,0.709519,0.096831,0.749943,0.089228,5
6,happy,Handcrafted,0.989749,0.015315,0.992052,0.012649,5
7,happy,emotion2vec,0.737439,0.051764,0.769238,0.048915,5
8,neutral,Handcrafted,0.986277,0.009616,0.989658,0.008326,5
9,neutral,emotion2vec,0.752453,0.109890,0.781763,0.105443,5



Saved fixed domain-probing outputs to: /content/drive/MyDrive/New Jurnal Cross/results_domain_probing
